In [11]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import seaborn as sns
import datetime
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_selection import chi2
import plotly.express as px
import shap

In [12]:
master_features_df = pd.read_csv("./test_features.csv")

In [13]:
reduced_features = master_features_df.dropna(subset=['Chi2'])
reduced_features = reduced_features.sort_values(by='Chi2', ascending=False).reset_index(drop=True)
reduced_features['Rank'] = np.arange(1, len(reduced_features)+1)


fig = px.line(reduced_features,
              x='Rank',
              y='Chi2',
              log_y=True,
              hover_name='Feature')

fig.show()

In [ ]:
print("Total features count: ", len(master_features_df),"\n")

relevant_features_df = master_features_df[
    ((master_features_df['Chi2'].notna()) & (master_features_df['Chi2'] > 0.68)) & #rank of about 500
    ((master_features_df['Lasso'] > 0.01) |
    (master_features_df['XGBoost'] > 0.01) |
    (master_features_df['DecisionTree'] > 0.01))
]

redundant_features = master_features_df[
    (master_features_df['Chi2'].fillna(0) < 0.33) &
    (master_features_df['Lasso'] <= 0.001) |
    (master_features_df['XGBoost'] <= 0.001) |
    (master_features_df['DecisionTree'] <= 0.001)
]

print(f"Number of features with Chi2 = 0 and no ML model importance (Redundant Features): {len(redundant_features)}","\n")

print(f"Number of features with Chi2_Average > 0 or at least some model importance (Relevant Features): {len(relevant_features_df)}","\n")

print("Maximum Lasso Importance for Redundant Features:")
print(redundant_features['Lasso'].max(),"\n")

print("Maximum XGBoost Importance for Redundant Features:")
print(redundant_features['XGBoost'].max(),"\n")

print("Maximum Decision Tree Importance for Redundant Features:")
print(redundant_features['DecisionTree'].max(),"\n") 

lasso_001 = redundant_features[redundant_features['Lasso'] > 0.0001]
print("Count of Redundant Features with Lasso Importance > 0.0001 (baseline assuming 10000 features):")
print(len(lasso_001),"\n")

print("Count of redundant features with XGBoost importance > 0.0001 (baseline assuming 10000 features):")
print(len(redundant_features[redundant_features['XGBoost'] > 0.0001]),"\n")

print("Count of redundant features with Decision Tree importance > 0.0001 (baseline assuming 10000 features):")
print(len(redundant_features[redundant_features['DecisionTree'] > 0.0001]),"\n")


Total features count:  8988 

Number of features with Chi2 = 0 and no ML model importance (Redundant Features): 8955 

Number of features with Chi2_Average > 0 or at least some model importance (Relevant Features): 282 

Maximum Lasso Importance for Redundant Features:
5.24252007634916 

Maximum XGBoost Importance for Redundant Features:
0.052603297 

Maximum Decision Tree Importance for Redundant Features:
0.0062171846175677 

Count of Redundant Features with Lasso Importance > 0.0001 (baseline assuming 10000 features):
314 

Count of redundant features with XGBoost importance > 0.0001 (baseline assuming 10000 features):
319 

Count of redundant features with Decision Tree importance > 0.0001 (baseline assuming 10000 features):
15 



In [15]:
#reduced_features_df = master_features_df.loc[(master_features_df['Lasso'] != 0)  & (master_features_df['DecisionTree'] != 0) & (master_features_df['XGBoost'] != 0)]

In [16]:
relevant_features_df.to_csv('top_reduced_features_2800_0.5_double_refined.csv', index=False)

In [17]:
# Create interactive log plot
fig = px.line(relevant_features_df, 
              x=range(1, len(relevant_features_df) + 1), 
              y='Chi2', 
              log_y=True,
              hover_name=relevant_features_df.index,
              title="Interactive Chi Squared Importance (Reduced Features)")

# Add a threshold line
fig.show()